### CAPSTONE: Multi-Agent Trip Planner By Syed Shaheryar Haider
**Agents**: 
- Itinerary Agent (LLM)
- Budget Agent (rule-based)
- Critic Agent (rule-based scorer)

**Framework**: OpenAI Agents SDK + a plain Python tool (weather API)


#### Project Overview
This notebook implements a multi-agent trip-planning system using the OpenAI Agents SDK.
Three agents collaborate to produce a budget-aware, weather-aware travel itinerary:

1. **Itinerary Agent** (LLM-powered) — generates a day-by-day plan and calls a live weather
   tool to avoid scheduling outdoor activities on poor-weather days.
2. **Budget Agent** (rule-based) — independently verifies the itinerary's total cost from
   structured data rather than trusting the LLM's self-reported arithmetic.
3. **Critic Agent** (rule-based) — scores the plan on budget fit and activity variety, and
   feeds natural-language feedback back into the Itinerary Agent for revision.


#### Setup 
pip install openai-agents python-dotenv requests

In [ ]:
import os
import json
import logging
import requests
from dataclasses import dataclass, field
from typing import Optional
import re
import asyncio

from agents import Agent, Runner, function_tool

os.environ["OPENAI_API_KEY"] = "Hidden"

In [18]:
# LOGGING SETUP (operational requirement: log agent interactions)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("trip_planner")

#### TOOL: Weather lookup (Open-Meteo, free, no API key needed)
This is the "tool use / environment interaction" requirement.

In [19]:
# Minimal city -> lat/lon lookup table (avoids needing a geocoding API key)
CITY_COORDS = {
    "paris": (48.8566, 2.3522),
    "tokyo": (35.6895, 139.6917),
    "new york": (40.7128, -74.0060),
    "rome": (41.9028, 12.4964),
    "barcelona": (41.3874, 2.1686),
    "lisbon": (38.7223, -9.1393),
    "bangkok": (13.7563, 100.5018),
}

@function_tool
def get_weather_forecast(city: str) -> str:
    """Get a short-term weather forecast for a city to help plan outdoor activities.

    Args:
        city: Name of the destination city (e.g., "Paris").
    """
    key = city.strip().lower()
    if key not in CITY_COORDS:
        logger.warning(f"[TOOL:get_weather_forecast] Unknown city '{city}', using mock data")
        return json.dumps({"city": city, "note": "No coordinates on file; assume mild, partly cloudy weather."})

    lat, lon = CITY_COORDS[key]
    try:
        url = (
            "https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}"
            "&daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max"
            "&timezone=auto&forecast_days=5"
        )
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        data = resp.json()
        logger.info(f"[TOOL:get_weather_forecast] Live forecast fetched for {city}")
        return json.dumps({"city": city, "forecast": data.get("daily", {})})
    except Exception as e:
        # Fallback so the system stays usable if the network call fails (resilience / guardrail)
        logger.warning(f"[TOOL:get_weather_forecast] API call failed ({e}); falling back to mock data")
        return json.dumps({"city": city, "note": "Forecast unavailable; assume mild, partly cloudy weather."})



#### SHARED STATE (the "common memory" the agents read/write to)

In [20]:
@dataclass
class TripState:
    destination: str
    days: int
    budget_usd: float
    preferences: str
    iteration: int = 0
    max_iterations: int = 3          # GUARDRAIL: caps the revision loop
    history: list = field(default_factory=list)  # logged interactions for the report

#### AGENT 1: Itinerary Agent (LLM-powered, uses the weather tool)

In [21]:
itinerary_agent = Agent(
    name="Itinerary Agent",
    instructions=(
        "You are a travel itinerary planner. Given a destination, number of days, "
        "budget, and traveler preferences, produce a day-by-day itinerary. "
        "Use the get_weather_forecast tool to check conditions. "
        "Write the human-readable itinerary as usual. "
        "Then, at the very end, output a single line in EXACTLY this format "
        "(no extra $ signs anywhere else in a way that could be confused with this line):\n"
        "ACTIVITY_COSTS_JSON: [12, 15, 0, 19, 20, 0, ...]\n"
        "This must be a JSON array of every individual activity's cost as a plain number "
        "(0 for free activities), one entry per activity, in the order they appear. "
        "Do not include subtotals or the grand total in this array — only individual activity costs."
    ),
    tools=[get_weather_forecast],
    model="gpt-4o-mini",
)

#### AGENT 2: Budget Agent (rule-based)

In [22]:
class BudgetAgent:
    """Rule-based agent: checks a proposed itinerary against the budget constraint."""

    name = "Budget Agent"

    def review(self, itinerary_text: str, budget_usd: float) -> dict:
        match = re.search(r"ACTIVITY_COSTS_JSON:\s*(\[[^\]]*\])", itinerary_text)
        if not match:
            return {"estimated_cost": None, "verdict": "UNKNOWN",
                    "message": "Could not find ACTIVITY_COSTS_JSON line."}

        try:
            costs = json.loads(match.group(1))
            estimated_cost = sum(float(c) for c in costs)
        except Exception:
            return {"estimated_cost": None, "verdict": "UNKNOWN",
                    "message": "Could not parse ACTIVITY_COSTS_JSON."}

        if estimated_cost <= budget_usd:
            verdict, message = "WITHIN_BUDGET", f"Computed cost ${estimated_cost:.2f} is within the ${budget_usd:.2f} budget."
        else:
            overage = estimated_cost - budget_usd
            verdict, message = "OVER_BUDGET", f"Computed cost ${estimated_cost:.2f} exceeds budget by ${overage:.2f}."

        result = {"estimated_cost": estimated_cost, "verdict": verdict, "message": message}
        logger.info(f"[Budget Agent] {result}")
        return result

#### AGENT 3: Critic Agent 
Rule-based scorer — drives the LEARNING/ADAPTATION loop: the score feeds back into the Itinerary Agent's next prompt, so it adapts its next proposal

In [23]:
class CriticAgent:
    """Rule-based agent: scores an itinerary on budget fit + activity variety."""

    name = "Critic Agent"

    def score(self, itinerary_text: str, budget_review: dict) -> dict:
        # Budget fit score (0-5)
        if budget_review["verdict"] == "WITHIN_BUDGET":
            budget_score = 5
        elif budget_review["verdict"] == "OVER_BUDGET":
            budget_score = 1
        else:
            budget_score = 2

        # Variety score (0-5): crude proxy = number of distinct activity keywords
        keywords = ["museum", "hike", "tour", "beach", "market", "restaurant", "park",
                    "temple", "show", "cruise", "class", "landmark", "gallery"]
        variety_hits = sum(1 for kw in keywords if kw in itinerary_text.lower())
        variety_score = min(5, variety_hits)

        total = budget_score + variety_score  # out of 10
        feedback = []
        if budget_score < 5:
            feedback.append("Reduce cost to fit budget.")
        if variety_score < 3:
            feedback.append("Add more variety in activity types (e.g., mix museums, outdoor, food).")

        result = {
            "budget_score": budget_score,
            "variety_score": variety_score,
            "total_score": total,
            "feedback": " ".join(feedback) if feedback else "Looks good.",
        }
        logger.info(f"[Critic Agent] {result}")
        return result

#### ORCHESTRATION LOOP
- Itinerary Agent <-> Budget Agent <-> Critic Agent, with the critic's feedback fed back into the Itinerary Agent's next prompt.
- GUARDRAIL: capped at state.max_iterations to prevent infinite loops.

In [24]:
async def plan_trip(state: TripState):
    budget_agent = BudgetAgent()
    critic_agent = CriticAgent()

    feedback_note = ""
    best_itinerary = None
    best_review = None
    best_score = None
    best_key = None  # lower is "better": (overage, -total_score)

    while state.iteration < state.max_iterations:
        state.iteration += 1
        logger.info(f"--- Iteration {state.iteration} ---")

        prompt = (
            f"Plan a {state.days}-day trip to {state.destination}. "
            f"Budget: ${state.budget_usd}. Preferences: {state.preferences}. "
        )
        if feedback_note:
            prompt += f"\n\nFeedback from previous round: {feedback_note}\nPlease revise accordingly."

        # --- Itinerary Agent runs (LLM reasoning + tool use) ---
        run_result = await Runner.run(itinerary_agent, prompt)
        itinerary_text = run_result.final_output
        state.history.append({"iteration": state.iteration, "role": "ItineraryAgent", "output": itinerary_text})

        # --- Budget Agent reviews (agent-to-agent: Itinerary -> Budget) ---
        budget_review = budget_agent.review(itinerary_text, state.budget_usd)
        budget_review["_budget"] = state.budget_usd
        state.history.append({"iteration": state.iteration, "role": "BudgetAgent", "output": budget_review})

        # --- Critic Agent scores (agent-to-agent: Budget -> Critic) ---
        score_result = critic_agent.score(itinerary_text, budget_review)
        state.history.append({"iteration": state.iteration, "role": "CriticAgent", "output": score_result})

        # --- Track the BEST iteration seen so far, not just the most recent one ---
        # Rank by budget overage first (closer to/under budget wins), then by total_score as a tiebreaker among plans with equal overage.
        estimated_cost = budget_review.get("estimated_cost")
        if estimated_cost is None:
            overage = float("inf")  # UNKNOWN verdict: treat as worst-case
        else:
            overage = max(0.0, estimated_cost - state.budget_usd)
        current_key = (overage, -score_result["total_score"])

        if best_key is None or current_key < best_key:
            best_key = current_key
            best_itinerary, best_review, best_score = itinerary_text, budget_review, score_result
            logger.info(f"New best plan found at iteration {state.iteration} (score={score_result['total_score']}, overage=${overage:.2f}).")

        # --- Stopping condition ---
        if score_result["total_score"] >= 8 and budget_review["verdict"] == "WITHIN_BUDGET":
            logger.info(f"Acceptable plan found after {state.iteration} iteration(s). Stopping.")
            break

        # --- Feed Critic's feedback back into the next Itinerary Agent prompt (LEARNING LOOP) ---
        feedback_note = (
            f"{budget_review['message']} {score_result['feedback']} "
            f"(Score: {score_result['total_score']}/10)"
        )

    if state.iteration >= state.max_iterations:
        logger.warning("Max iterations reached without an ideal plan; returning best effort across all iterations.")

    return {
        "itinerary": best_itinerary,
        "budget_review": best_review,
        "score": best_score,
        "iterations_used": state.iteration,
        "history": state.history,
    }

#### Run the System: Three Budget Scenarios
We test the trip planner under three budget conditions (comfortable, tight, very tight) to observe how the feedback loop behaves under different constraints.

*(Note: the three scenario cells below each call `plan_trip(state)` directly —
that's the actual entry point into the system. A single `main()` wrapper isn't
needed since we're comparing multiple budget scenarios side by side.)*

##### Scenario 1: Comfortable Budget ($600)
A generous budget for a 3-day Paris trip which is expected to converge quickly with little need for revision.

In [25]:
state_600 = TripState(
    destination="Paris",
    days=3,
    budget_usd=600,
    preferences="loves art, food, and walkable neighborhoods; not interested in nightlife",
)
result_600 = await plan_trip(state_600)

print("\n========== FINAL ITINERARY ($600) ==========\n")
print(result_600["itinerary"])
print("\n========== BUDGET REVIEW ==========")
print(result_600["budget_review"])
print("\n========== CRITIC SCORE ==========")
print(result_600["score"])
print(f"\nIterations used: {result_600['iterations_used']} / {state_600.max_iterations}")

with open("trip_planner_log_600.json", "w") as f:
    json.dump(result_600["history"], f, indent=2, default=str)
print("Saved to trip_planner_log_600.json")

2026-06-21 14:46:28,462 | trip_planner | INFO | --- Iteration 1 ---
2026-06-21 14:46:29,539 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:46:30,216 | trip_planner | INFO | [TOOL:get_weather_forecast] Live forecast fetched for Paris
2026-06-21 14:46:40,214 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:46:40,217 | trip_planner | INFO | [Budget Agent] {'estimated_cost': 202.0, 'verdict': 'WITHIN_BUDGET', 'message': 'Computed cost $202.00 is within the $600.00 budget.'}
2026-06-21 14:46:40,218 | trip_planner | INFO | [Critic Agent] {'budget_score': 5, 'variety_score': 4, 'total_score': 9, 'feedback': 'Looks good.'}
2026-06-21 14:46:40,218 | trip_planner | INFO | New best plan found at iteration 1 (score=9, overage=$0.00).
2026-06-21 14:46:40,219 | trip_planner | INFO | Acceptable plan found after 1 iteration(s). Stopping.



========== FINAL ITINERARY ($600) ==========

Here's a 3-day itinerary for your trip to Paris, focused on art, food, and walkable neighborhoods. The weather looks warm, so be prepared for the heat!

### Day 1: Arrival and Montmartre Exploration
- **Morning**: Arrive in Paris. Check into your hotel in the Montmartre area, known for its artistic history and charming streets.
- **Lunch**: Grab a meal at a local café. Try the classic French onion soup and croque monsieur. (~$15)
- **Afternoon**: Visit the **Sacré-Cœur Basilica** for stunning views of the city. Entry is free! Explore the charming streets of Montmartre, dotted with artists.
- **Snack**: Stop by a patisserie for a freshly baked croissant or pain au chocolat. (~$5)
- **Evening**: Dinner at a nearby bistro to sample traditional French cuisine. (~$25)
  
### Day 2: Louvre and Seine River Walk
- **Morning**: Head to the **Louvre Museum** for a morning filled with art. Tip: Buy tickets in advance to avoid long lines. (~$20)
- **L

##### Scenario 2: Tight Budget ($150)
A constrained but feasible budget tests whether the agent can plan efficiently without much slack.

In [26]:
state_150 = TripState(
    destination="Paris",
    days=3,
    budget_usd=150,
    preferences="loves art, food, and walkable neighborhoods; not interested in nightlife",
)
result_150 = await plan_trip(state_150)

print("\n========== FINAL ITINERARY ($150) ==========\n")
print(result_150["itinerary"])
print("\n========== BUDGET REVIEW ==========")
print(result_150["budget_review"])
print("\n========== CRITIC SCORE ==========")
print(result_150["score"])
print(f"\nIterations used: {result_150['iterations_used']} / {state_150.max_iterations}")

with open("trip_planner_log_150.json", "w") as f:
    json.dump(result_150["history"], f, indent=2, default=str)
print("Saved to trip_planner_log_150.json")

2026-06-21 14:46:40,230 | trip_planner | INFO | --- Iteration 1 ---
2026-06-21 14:46:41,387 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:46:42,081 | trip_planner | INFO | [TOOL:get_weather_forecast] Live forecast fetched for Paris
2026-06-21 14:46:52,105 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:46:52,108 | trip_planner | INFO | [Budget Agent] {'estimated_cost': 138.0, 'verdict': 'WITHIN_BUDGET', 'message': 'Computed cost $138.00 is within the $150.00 budget.'}
2026-06-21 14:46:52,110 | trip_planner | INFO | [Critic Agent] {'budget_score': 5, 'variety_score': 2, 'total_score': 7, 'feedback': 'Add more variety in activity types (e.g., mix museums, outdoor, food).'}
2026-06-21 14:46:52,111 | trip_planner | INFO | New best plan found at iteration 1 (score=7, overage=$0.00).
2026-06-21 14:46:52,111 | trip_planner | INFO | --- Iteration 2 ---
2026-06-21 14:46:53,925 | htt


========== FINAL ITINERARY ($150) ==========

### 3-Day Itinerary for Paris

#### Overview
- **Budget:** $150
- **Focus:** Art, food, walkable neighborhoods
- **Weather Forecast:** Expect hot weather with highs around 36-39°C and low chances of rain.

---

### Day 1: Art & Neighborhood Stroll

**Morning**
- **Visit the Louvre Museum**  
   *Cost:* $17   
   Explore the world’s largest art museum and a historic monument in Paris. Arrive early to avoid long lines.  
   **Duration:** 3 hours

**Afternoon**
- **Lunch at Café Marly**  
   *Cost:* $25  
   Enjoy a meal with views of the Louvre glass pyramid.  
   **Duration:** 1.5 hours

- **Stroll through the Tuileries Garden**  
   *Cost:* Free  
   Enjoy the beautiful gardens and sculptures. Perfect for relaxing after lunch.  
   **Duration:** 1 hour

**Evening**
- **Visit Musée de l'Orangerie**  
   *Cost:* $12  
   Home to Monet's Water Lilies, a serene setting not overly crowded.  
   **Duration:** 1.5 hours

---
  
### Day 2: Art, Fo

##### Scenario 3: Very Tight Budget ($80)
An aggressively constrained budget designed to stress-test the feedback loop and observe the Critic-driven revision process across multiple iterations.

In [27]:
state_80 = TripState(
    destination="Paris",
    days=3,
    budget_usd=80,
    preferences="loves art, food, and walkable neighborhoods; not interested in nightlife",
)
result_80 = await plan_trip(state_80)

print("\n========== FINAL ITINERARY ($80) ==========\n")
print(result_80["itinerary"])
print("\n========== BUDGET REVIEW ==========")
print(result_80["budget_review"])
print("\n========== CRITIC SCORE ==========")
print(result_80["score"])
print(f"\nIterations used: {result_80['iterations_used']} / {state_80.max_iterations}")

with open("trip_planner_log_80.json", "w") as f:
    json.dump(result_80["history"], f, indent=2, default=str)
print("Saved to trip_planner_log_80.json")

2026-06-21 14:47:08,613 | trip_planner | INFO | --- Iteration 1 ---
2026-06-21 14:47:09,935 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:47:10,633 | trip_planner | INFO | [TOOL:get_weather_forecast] Live forecast fetched for Paris
2026-06-21 14:47:24,366 | httpx | INFO | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-21 14:47:24,369 | trip_planner | INFO | [Budget Agent] {'estimated_cost': 126.0, 'verdict': 'OVER_BUDGET', 'message': 'Computed cost $126.00 exceeds budget by $46.00.'}
2026-06-21 14:47:24,370 | trip_planner | INFO | [Critic Agent] {'budget_score': 1, 'variety_score': 4, 'total_score': 5, 'feedback': 'Reduce cost to fit budget.'}
2026-06-21 14:47:24,370 | trip_planner | INFO | New best plan found at iteration 1 (score=5, overage=$46.00).
2026-06-21 14:47:24,370 | trip_planner | INFO | --- Iteration 2 ---
2026-06-21 14:47:25,907 | httpx | INFO | HTTP Request: POST https://api.openai


========== FINAL ITINERARY ($80) ==========

Here's a revised 3-day itinerary for your trip to Paris, keeping your budget of $80 in mind while focusing on art, food, and walkable neighborhoods:

### Day 1: Iconic Landmarks and Art
- **Morning**: Start your day with a visit to the **Sacré-Cœur Basilica** (Free entry). Explore the charming streets of Montmartre and enjoy the local art scene.
- **Lunch**: Stop by a local café for a classic **French baguette sandwich**. Approximate cost: $8.
- **Afternoon**: Stroll through **Bastille Market** (free to explore) and enjoy browsing the food stalls filled with fresh produce and artisanal goods.
- **Dinner**: Enjoy an affordable meal at **Le Petit Cler** where you can savor classic **French comfort food**. Approximate cost: $15.

### Day 2: Art and Culture
- **Morning**: Visit the **Musée de l'Orangerie** to see Monet's Water Lilies. Entry fee: $12.
- **Lunch**: Grab a quick bite from a nearby street vendor for a **crepe**. Approximate cost: $

##### Inspect Iteration-by-Iteration History (Scenario 3)
Breaks down each round of the $80 run to show how the Itinerary Agent revised its plan in response to the Budget and Critic agents' feedback.

In [28]:
for entry in result_80["history"]:
    print(f"--- Iteration {entry['iteration']} | {entry['role']} ---")
    if entry['role'] == "ItineraryAgent":
        text = entry['output']
        print(text[:400], "...\n")

        total_match = re.search(r"(Total Cost|Grand Total)[:\*\s]*\$?(\d+(?:\.\d+)?)", text)
        costs_match = re.search(r"ACTIVITY_COSTS_JSON:\s*(\[[^\]]*\])", text)
        if total_match:
            print(f"  [LLM self-reported total: ${total_match.group(2)}]")
        if costs_match:
            costs = json.loads(costs_match.group(1))
            print(f"  [Parsed activity costs: {costs} -> sums to ${sum(costs):.2f}]")
        print()
    else:
        print(entry['output'], "\n")

--- Iteration 1 | ItineraryAgent ---
Here's a 3-day itinerary for your trip to Paris, focusing on art, food, and walkable neighborhoods. The weather looks warm, so bring along light clothing and stay hydrated!

### Day 1: Explore Montmartre

**Morning:**
- **Sacré-Cœur Basilica:** Start your day with a visit to this stunning basilica. Enjoy the panoramic view of the city.  
  *Cost: Free*

**Late Morning:**
- **Montmartre Walking Tou ...

  [Parsed activity costs: [0, 0, 15, 12, 25, 19, 10, 0, 15, 10, 0, 20] -> sums to $126.00]

--- Iteration 1 | BudgetAgent ---
{'estimated_cost': 126.0, 'verdict': 'OVER_BUDGET', 'message': 'Computed cost $126.00 exceeds budget by $46.00.', '_budget': 80} 

--- Iteration 1 | CriticAgent ---
{'budget_score': 1, 'variety_score': 4, 'total_score': 5, 'feedback': 'Reduce cost to fit budget.'} 

--- Iteration 2 | ItineraryAgent ---
Here's a revised 3-day itinerary for your trip to Paris, keeping your budget of $80 in mind while focusing on art, food, and wa